In [ ]:
import pandas as pd
from datasets import Dataset
from tqdm import tqdm

DATA_FILE = "/content/drive/MyDrive/summerize/easc/EASC.csv"
OUTPUT_FILE = "/content/drive/MyDrive/summerize/easc/EASC_with_lengths.csv"

def load_csv(path):
    data = pd.read_csv(path, encoding="utf-8")

    # Ensure columns exist
    if "text_x" not in data.columns or "text_y" not in data.columns:
        raise ValueError("CSV must contain 'text_x' and 'text_y' columns.")

    # Calculate text lengths (number of words)
    data["text_length_x"] = data["text_x"].apply(lambda x: len(str(x).split()))
    data["text_length_y"] = data["text_y"].apply(lambda x: len(str(x).split()))

    # Compute medians
    median_x = data["text_length_x"].median()
    median_y = data["text_length_y"].median()
    print(f"📝 Median text length (text_x): {median_x} words")
    print(f"📝 Median summary length (text_y): {median_y} words")

    # Categorize length relative to median
    def length_category(length, median):
        if length < median * 0.75:
            return "short"
        elif length <= median * 1.25:
            return "medium"
        else:
            return "long"

    data["length_category_x"] = data["text_length_x"].apply(lambda l: length_category(l, median_x))
    data["length_category_y"] = data["text_length_y"].apply(lambda l: length_category(l, median_y))

    # Convert to Hugging Face Dataset
    data_dict = {
        "text_x": [], "text_y": [],
        "text_length_x": [], "length_category_x": [],
        "text_length_y": [], "length_category_y": []
    }
    for _, row in tqdm(data.iterrows(), total=len(data), desc="Loading dataset"):
        data_dict["text_x"].append(row["text_x"])
        data_dict["text_y"].append(row["text_y"])
        data_dict["text_length_x"].append(row["text_length_x"])
        data_dict["length_category_x"].append(row["length_category_x"])
        data_dict["text_length_y"].append(row["text_length_y"])
        data_dict["length_category_y"].append(row["length_category_y"])

    dataset = Dataset.from_dict(data_dict)
    return dataset, data

# Load dataset and compute lengths
dataset, updated_df = load_csv(DATA_FILE)
print(f"📊 Dataset size: {len(dataset)}")
print(updated_df.head())

# Save to new CSV
updated_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"✅ Updated dataset saved as {OUTPUT_FILE}")


📝 Median text length (text_x): 343.0 words
📝 Median summary length (text_y): 112.0 words


Loading dataset: 100%|██████████| 153/153 [00:00<00:00, 8957.06it/s]

📊 Dataset size: 153
   id                                             text_x  \
0   1  لودفيج فان بيتهوفن مؤلف موسيقي ألماني ولد عام ...   
1   2  الناي آلة نفخية تعد بحق أقدم آلة موسيقية في ال...   
2   3  العود هي آلة موسيقية شرقية وترية تاريخها موغل ...   
3   4   الخطاط الكويتي فريد العلي يعرض مجموعة محمديات...   
4   5  لفت نظري واستغرابي إعلانان كلاهما يتعلق بـ"طلب...   

                                              text_y  text_length_x  \
0   له الفضل الاعظم في تطوير الموسيقى الكلاسيكية....            436   
1  الناي آلة نفخية تعد بحق اقدم آلة موسيقية في ال...            371   
2  العود هي آلة موسيقية شرقية وترية تاريخها موغل ...            153   
3   وفي اليوم الثاني من المعرض اقام الفنان العلي ...            153   
4  لفت نظري واستغرابي اعلانان كلاهما يتعلق بـ"طلب...            506   

   text_length_y length_category_x length_category_y  
0             68              long             short  
1            231            medium              long  
2             25           

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0c951983f47b0c257a145eac443616a017ba9e7aaaa9d56d7239faecb76f383a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
# finetune_arabic_summarizer_hierarchical.py

import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed
)

# ======================
# CONFIG
# ======================
MODEL_NAME = "malmarjeh/t5-arabic-text-summarization"
DATA_FILE  = "/content/drive/MyDrive/summerize/easc/EASC.csv"
OUTPUT_DIR = "/content/drive/MyDrive/summerize/t5-arabic-hierarchical"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 2
EPOCHS = 5
LR = 3e-5
SEED = 42
BETA_SALIENCE = 0.3   # weight of auxiliary loss

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# LOAD DATA
# ======================
def load_csv(path):
    df = pd.read_csv(path)
    data = {"text": [], "summary": []}

    for _, r in tqdm(df.iterrows(), total=len(df)):
        t, s = str(r["text_x"]).strip(), str(r["text_y"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)

    return Dataset.from_dict(data)

dataset = load_csv(DATA_FILE)

# ======================
# TOKENIZER
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prompt(x):
    return [f"النص: {t}\nالملخص:" for t in x]

# ======================
# WEAK SALIENCE LABELS
# ======================
def build_salience(input_ids, summary_ids):
    summary_set = set([i for i in summary_ids if i != tokenizer.pad_token_id])
    return [1 if tok in summary_set else 0 for tok in input_ids]

# ======================
# PREPROCESS
# ======================
def preprocess(examples):
    inputs  = prompt(examples["text"])
    targets = examples["summary"]

    enc = tokenizer(inputs, max_length=MAX_INPUT_LENGTH,
                    truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        dec = tokenizer(targets, max_length=MAX_TARGET_LENGTH,
                        truncation=True, padding="max_length")

    labels = []
    salience = []

    for inp, lab in zip(enc["input_ids"], dec["input_ids"]):
        lab_masked = [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        labels.append(lab_masked)
        salience.append(build_salience(inp, lab))

    enc["labels"] = labels
    enc["salience_labels"] = salience
    return enc

dataset_tok = dataset.map(preprocess, batched=True,
                          remove_columns=dataset.column_names)

# ======================
# HIERARCHICAL + ADAPTER MODEL
# ======================
class HierarchicalT5(nn.Module):
    def __init__(self, name, adapter_dim=256):
        super().__init__()
        self.base = AutoModelForSeq2SeqLM.from_pretrained(name)
        d = self.base.config.d_model

        # adapter
        self.adapter = nn.Sequential(
            nn.Linear(d, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, d)
        )

        # sentence context projection
        self.sent_proj = nn.Linear(d, d)

        # salience head
        self.salience_head = nn.Linear(d, 1)

        self.base.config.dropout_rate = 0.2
        self.base.config.attention_dropout_rate = 0.2
        self.base.config.num_beams = 4
        self.base.config.min_length = 10
        self.base.config.max_length = MAX_TARGET_LENGTH

    def sentence_context(self, hidden, attention_mask):
        # simple sentence pooling by chunks of 20 tokens
        B,T,D = hidden.shape
        chunk = 20
        ctx = torch.zeros_like(hidden)

        for i in range(0, T, chunk):
            h = hidden[:, i:i+chunk, :]               # [B,c,D]
            m = attention_mask[:, i:i+chunk].unsqueeze(-1)
            pooled = (h*m).sum(1) / (m.sum(1)+1e-8)    # [B,D]
            pooled = self.sent_proj(pooled).unsqueeze(1)
            ctx[:, i:i+chunk, :] = pooled

        return ctx

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, salience_labels=None, **kwargs):

        enc_out = self.base.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        H = enc_out.last_hidden_state                    # [B,T,D]

        # hierarchical sentence context
        H = H + self.sentence_context(H, attention_mask)

        # adapter residual
        H = H + self.adapter(H)

        # salience prediction
        salience_logits = self.salience_head(H).squeeze(-1)  # [B,T]

        outputs = self.base(
            input_ids=None,
            attention_mask=attention_mask,
            encoder_outputs=(H,),
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

        if salience_labels is not None:
            salience_labels = salience_labels.float()
            sal_loss = F.binary_cross_entropy_with_logits(
                salience_logits, salience_labels)
            loss = loss + BETA_SALIENCE * sal_loss

        outputs.loss = loss
        return outputs

    def generate(self, *args, **kwargs):
        return self.base.generate(*args, **kwargs)

model = HierarchicalT5(MODEL_NAME)

# ======================
# DATA COLLATOR
# ======================
class CustomCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        sal = torch.tensor([f.pop("salience_labels") for f in features])
        batch = super().__call__(features)
        batch["salience_labels"] = sal
        return batch

collator = CustomCollator(tokenizer, model=model.base, label_pad_token_id=-100)

# ======================
# TRAINING ARGS
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

# ======================
# TRAINER
# ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=dataset_tok,
    tokenizer=tokenizer,
    data_collator=collator
)

print("🚀 Training hierarchical Arabic summarizer...")
trainer.train()

print("💾 Saving...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Done:", OUTPUT_DIR)


Loading CSV: 100%|██████████| 100/100 [00:00<00:00, 2687.93it/s]


🔄 Loading tokenizer...
🔄 Loading model with adapter...
🔧 Tokenizing dataset...


Tokenizing:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-291964462.py:204: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


🚀 Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,6.259600


KeyboardInterrupt: 

In [2]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt

nltk.download("wordnet")

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed
)

# ======================
# CONFIG
# ======================
MODEL_NAME = "google/mt5-small"
DATA_FILE  = "xl_sum.csv"
OUTPUT_DIR = "t5-arabic-hierarchical-metrics4"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4
EPOCHS = 15
LR = 1e-3
SEED = 42
BETA_SALIENCE = 0.3

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# LOAD DATA
# ======================
def load_csv(path):
    df = pd.read_csv(path)
    data = {"text": [], "summary": []}
    for _, r in tqdm(df.iterrows(), total=len(df)):
        t, s = str(r["text"]).strip(), str(r["summary"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)
    return Dataset.from_dict(data)

dataset = load_csv(DATA_FILE)

# ======================
# TOKENIZER
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prompt(x):
    return [f"النص: {t}\nالملخص:" for t in x]

# ======================
# WEAK SALIENCE LABELS
# ======================
def build_salience(input_ids, summary_ids):
    summary_set = set([i for i in summary_ids if i != tokenizer.pad_token_id])
    return [1 if tok in summary_set else 0 for tok in input_ids]

# ======================
# PREPROCESS
# ======================
def preprocess(examples):
    inputs  = prompt(examples["text"])
    targets = examples["summary"]

    enc = tokenizer(inputs, max_length=MAX_INPUT_LENGTH,
                    truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        dec = tokenizer(targets, max_length=MAX_TARGET_LENGTH,
                        truncation=True, padding="max_length")

    labels, salience = [], []

    for inp, lab in zip(enc["input_ids"], dec["input_ids"]):
        lab_masked = [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        labels.append(lab_masked)
        salience.append(build_salience(inp, lab))

    enc["labels"] = labels
    enc["salience_labels"] = salience
    return enc

dataset_tok = dataset.map(preprocess, batched=True,
                          remove_columns=dataset.column_names)

# train/validation split
dataset_tok = dataset_tok.train_test_split(test_size=0.1, seed=SEED)
train_ds = dataset_tok["train"]
eval_ds  = dataset_tok["test"]

# ======================
# HIERARCHICAL T5 MODEL
# ======================
class HierarchicalT5(nn.Module):
    def __init__(self, name, adapter_dim=256):
        super().__init__()
        self.base = AutoModelForSeq2SeqLM.from_pretrained(name)
        d = self.base.config.d_model

        self.adapter = nn.Sequential(
            nn.Linear(d, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, d)
        )

        self.sent_proj = nn.Linear(d, d)
        self.salience_head = nn.Linear(d, 1)

        self.base.config.num_beams = 4
        self.base.config.min_length = 10
        self.base.config.max_length = MAX_TARGET_LENGTH

    @property
    def generation_config(self):
        return self.base.generation_config

    def sentence_context(self, hidden, attention_mask):
        B,T,D = hidden.shape
        chunk = 20
        ctx = torch.zeros_like(hidden)

        for i in range(0, T, chunk):
            h = hidden[:, i:i+chunk, :]
            m = attention_mask[:, i:i+chunk].unsqueeze(-1)
            pooled = (h*m).sum(1) / (m.sum(1)+1e-8)
            pooled = self.sent_proj(pooled).unsqueeze(1)
            ctx[:, i:i+chunk, :] = pooled

        return ctx

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, salience_labels=None, **kwargs):

        enc_out = self.base.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        H = enc_out.last_hidden_state
        H = H + self.sentence_context(H, attention_mask)
        H = H + self.adapter(H)

        salience_logits = self.salience_head(H).squeeze(-1)

        outputs = self.base(
            input_ids=None,
            attention_mask=attention_mask,
            encoder_outputs=(H,),
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

        if salience_labels is not None:
            salience_labels = salience_labels.float()
            sal_loss = F.binary_cross_entropy_with_logits(
                salience_logits, salience_labels)
            loss = loss + BETA_SALIENCE * sal_loss

        outputs.loss = loss
        return outputs

    def generate(self, *args, **kwargs):
        filtered_kwargs = {k: v for k, v in kwargs.items() if k != 'salience_labels'}
        return self.base.generate(*args, **filtered_kwargs)
                               


model = HierarchicalT5(MODEL_NAME)

# ======================
# DATA COLLATOR
# ======================
class CustomCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        sal = torch.tensor([f.pop("salience_labels") for f in features])
        batch = super().__call__(features)
        batch["salience_labels"] = sal
        return batch

collator = CustomCollator(tokenizer, model=model.base, label_pad_token_id=-100)

# ======================
# METRICS
# ======================
bertscore_metric = evaluate.load("bertscore")
chrf_metric      = evaluate.load("chrf")
bleu_metric      = evaluate.load("bleu")
meteor_metric    = evaluate.load("meteor")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BERTScore
    bertscore = bertscore_metric.compute(predictions=decoded_preds,
                                         references=decoded_labels,
                                         lang="ar")

    # chrF
    chrf = chrf_metric.compute(predictions=decoded_preds,
                               references=decoded_labels)

    # BLEU
    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    # METEOR
    meteor = meteor_metric.compute(predictions=decoded_preds,
                                   references=decoded_labels)

    return {
        "bertscore_precision": np.mean(bertscore["precision"]),
        "bertscore_recall":    np.mean(bertscore["recall"]),
        "bertscore_f1":        np.mean(bertscore["f1"]),
        "chrf": chrf["score"],
        "bleu": bleu["bleu"],
        "meteor": meteor["meteor"],
    }

# ======================
# TRAINING ARGS
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    fp16=False,  # disable GPU fp16
    report_to="none",
    seed=SEED,
    save_safetensors=False
)

# ======================
# TRAINER
# ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

print("🚀 Training...")
trainer.train()

# ======================
# FINAL METRICS + PERPLEXITY
# ======================
metrics = trainer.evaluate()
if "eval_loss" in metrics and metrics["eval_loss"] is not None:
    metrics["perplexity"] = math.exp(metrics["eval_loss"])

print("Final metrics:", metrics)

# ======================
# SAVE MODEL
# ======================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ======================
# PLOTS
# ======================
log_history = trainer.state.log_history

train_loss, eval_loss = [], []
bertscore_f1, chrf_scores, bleu_scores = [], [], []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])
        bertscore_f1.append(log.get("eval_bertscore_f1", 0))
        chrf_scores.append(log.get("eval_chrf", 0))
        bleu_scores.append(log.get("eval_bleu", 0))

epochs_range = range(1, len(eval_loss)+1)

plt.figure(); plt.plot(train_loss); plt.title("Training Loss"); plt.show()
plt.figure(); plt.plot(epochs_range, eval_loss); plt.title("Validation Loss"); plt.show()
plt.figure(); plt.plot(epochs_range, bertscore_f1); plt.title("BERTScore F1"); plt.show()
plt.figure(); plt.plot(epochs_range, chrf_scores); plt.title("chrF"); plt.show()
plt.figure(); plt.plot(epochs_range, bleu_scores); plt.title("BLEU"); plt.show()

print("✅ Done. Model and plots saved.")


[nltk_data] Downloading package wordnet to /home/kawkab/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9378/9378 [00:00<00:00, 27348.28it/s]
/singularity/00-gupyter2/.ve-jhub/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map:   0%|                                                                                                                            

🚀 Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# evalu based on lenght

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
DYNAMIC ARABIC SUMMARIZATION PIPELINE
Zero hardcoded values — fully configurable via JSON/YAML/CLI
✅ FIXED: Valid public model by default + robust local path detection
"""

import re
import json
import yaml
import argparse
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field, asdict
from abc import ABC, abstractmethod
import sys
import os

# ======================
# 1. CONFIGURATION SYSTEM (Core)
# ======================
@dataclass
class ArabicNormalizationConfig:
    """Dynamic character/diacritic normalization rules"""
    remove_diacritics: bool = True
    diacritic_patterns: List[str] = field(default_factory=lambda: [
        r'[\u0617-\u061A\u064B-\u0652\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]'
    ])
    normalize_chars: bool = True
    char_map: Dict[str, str] = field(default_factory=lambda: {
        'أ': 'ا', 'إ': 'ا', 'آ': 'ا', 'ى': 'ي', 'ة': 'ه',
        'گ': 'ك', 'ڤ': 'ف', 'پ': 'ب', 'چ': 'ج'
    })
    remove_tatweel: bool = True
    tatweel_pattern: str = r'ـ+'
    remove_latin: bool = False
    extra_patterns: List[Dict[str, str]] = field(default_factory=list)  # [{"pattern": "...", "replacement": "..."}]

@dataclass
class SentenceSegmentationConfig:
    """Dynamic sentence boundary detection"""
    delimiters: List[str] = field(default_factory=lambda: ['.', '،', '!', '؟', '؛', ':'])
    min_sentence_length: int = 10
    preserve_order: bool = True

@dataclass
class ExtractiveRankingConfig:
    """Configurable graph-based sentence ranking"""
    ratio: float = 0.4
    min_sentences: int = 2
    similarity_threshold: float = 0.6
    triangle_weight: float = 0.3
    degree_weight: float = 0.7
    embedding_model: str = "UBC-NLP/MARBERTv2"
    max_embedding_length: int = 128

@dataclass
class AbstractiveGenerationConfig:
    """Fully dynamic generation parameters"""
    # ✅ FIXED: Using valid public model instead of invalid local path
    model_name: str = "/content/drive/MyDrive/summerize/easc/t5-arabic-hybrid-summarizer"  # Public HF model (works out-of-the-box)
    max_length: int = 150
    min_length_ratio: float = 0.3
    length_penalty: float = 1.2
    num_beams: int = 4
    temperature: float = 0.7
    no_repeat_ngram_size: int = 2
    early_stopping: bool = True
    prompt_template: str = """النص: {text}

التعليمات:
1. حدد الموضوع الرئيسي
2. استخرج الكيانات الأساسية
3. اكتب ملخصًا موجزًا ({min_sentences}-{max_sentences} جمل) بالفصحى
4. ركّز على الجوهر دون تفاصيل ثانوية

الملخص:"""
    prompt_params: Dict[str, Any] = field(default_factory=lambda: {
        "min_sentences": 2,
        "max_sentences": 3
    })
    force_msa: bool = True

@dataclass
class AugmentationConfig:
    """Dynamic augmentation pipeline"""
    strategies: List[Dict[str, Any]] = field(default_factory=lambda: [
        {"name": "vowel_deletion", "probability": 0.7, "enabled": True},
        {"name": "character_swap", "probability": 0.1, "enabled": True},
        {"name": "synonym_replacement", "probability": 0.2, "enabled": True}
    ])
    synonym_map: Dict[str, List[str]] = field(default_factory=lambda: {
        'كبير': ['عظيم', 'ضخم'], 'صغير': ['ضئيل', 'بسيط'],
        'جيد': ['ممتاز', 'رائع'], 'سيء': ['رديء', 'ضعيف']
    })

@dataclass
class PostProcessingConfig:
    """Configurable output refinement"""
    remove_incomplete_sentences: bool = True
    incomplete_sentence_pattern: str = r'[\u0600-\u06FF]+[^\u0600-\u06FF.،!؟؛:]*$'
    dialect_to_msa: Dict[str, str] = field(default_factory=lambda: {
        'عايز': 'يريد', 'بص': 'انظر', 'يالا': 'هيّا'
    })
    punctuation_fixes: List[Dict[str, str]] = field(default_factory=lambda: [
        {"pattern": r'\s+([.،!؟؛:,])', "replacement": r'\1'},
        {"pattern": r'([.،!؟؛:,])\s+', "replacement": r'\1 '}
    ])
    min_summary_length: int = 15

@dataclass
class PipelineConfig:
    """Master configuration container"""
    normalization: ArabicNormalizationConfig = field(default_factory=ArabicNormalizationConfig)
    segmentation: SentenceSegmentationConfig = field(default_factory=SentenceSegmentationConfig)
    extractive: ExtractiveRankingConfig = field(default_factory=ExtractiveRankingConfig)
    abstractive: AbstractiveGenerationConfig = field(default_factory=AbstractiveGenerationConfig)
    augmentation: AugmentationConfig = field(default_factory=AugmentationConfig)
    postprocessing: PostProcessingConfig = field(default_factory=PostProcessingConfig)
    device: str = "auto"  # auto, cpu, cuda, mps
    hybrid_mode: bool = True
    cache_dir: Optional[str] = None
    seed: int = 42

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "PipelineConfig":
        field_types = {f.name: f.type for f in cls.__dataclass_fields__.values()}
        cleaned = {}
        for k, v in data.items():
            if k not in cls.__dataclass_fields__:
                continue
            field_type = field_types[k]
            if hasattr(field_type, '__dataclass_fields__') and isinstance(v, dict):
                nested_cls = field_type
                nested_fields = {nk: nv for nk, nv in v.items() if nk in nested_cls.__dataclass_fields__}
                cleaned[k] = nested_cls(**nested_fields)
            else:
                cleaned[k] = v
        return cls(**cleaned)

    @classmethod
    def from_file(cls, path: str) -> "PipelineConfig":
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Config file not found: {path}")

        with open(path, 'r', encoding='utf-8') as f:
            if path.suffix in ['.yaml', '.yml']:
                data = yaml.safe_load(f)
            else:  # .json
                data = json.load(f)
        return cls.from_dict(data)

    def save(self, path: str):
        path = Path(path)
        data = self.to_dict()
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            if path.suffix in ['.yaml', '.yml']:
                yaml.dump(data, f, allow_unicode=True, sort_keys=False)
            else:
                json.dump(data, f, ensure_ascii=False, indent=2)


# ======================
# 2. DYNAMIC PREPROCESSING ENGINE
# ======================
class DynamicArabicPreprocessor:
    """100% configurable text normalization"""

    def __init__(self, config: ArabicNormalizationConfig):
        self.config = config
        self.compiled_patterns = []
        for pat in self.config.diacritic_patterns:
            self.compiled_patterns.append(re.compile(pat))
        for extra in self.config.extra_patterns:
            self.compiled_patterns.append(re.compile(extra['pattern']))

    def normalize(self, text: str) -> str:
        if not text or not isinstance(text, str):
            return ""

        result = text

        # Diacritics removal
        if self.config.remove_diacritics:
            for pattern in self.compiled_patterns:
                result = pattern.sub('', result)

        # Character normalization
        if self.config.normalize_chars:
            result = ''.join([self.config.char_map.get(c, c) for c in result])

        # Tatweel removal
        if self.config.remove_tatweel:
            result = re.sub(self.config.tatweel_pattern, '', result)

        # Latin removal
        if self.config.remove_latin:
            result = re.sub(r'[a-zA-Z0-9]', '', result)

        # Extra patterns
        for extra in self.config.extra_patterns:
            result = re.sub(extra['pattern'], extra['replacement'], result)

        # Whitespace cleanup
        result = re.sub(r'\s+', ' ', result).strip()

        return result

    def segment_sentences(self, text: str, config: SentenceSegmentationConfig) -> List[str]:
        if not text:
            return []

        # Build regex from delimiters
        delim_pattern = '|'.join(re.escape(d) for d in config.delimiters)
        pattern = f'(?<=[{delim_pattern}])\\s+'

        sentences = re.split(pattern, text.strip())
        filtered = [s.strip() for s in sentences if len(s.strip()) >= config.min_sentence_length]

        return filtered if config.preserve_order else sorted(filtered, key=len, reverse=True)


# ======================
# 3. DYNAMIC AUGMENTATION FACTORY
# ======================
class AugmentationStrategy(ABC):
    @abstractmethod
    def apply(self, text: str, config: Dict[str, Any], rng: np.random.Generator) -> str:
        pass

class VowelDeletionAugmentation(AugmentationStrategy):
    def apply(self, text: str, config: Dict[str, Any], rng: np.random.Generator) -> str:
        if rng.random() < config.get('probability', 0.7):
            return re.sub(r'[\u064B-\u0652]', '', text)
        return text

class CharacterSwapAugmentation(AugmentationStrategy):
    def apply(self, text: str, config: Dict[str, Any], rng: np.random.Generator) -> str:
        chars = list(text)
        prob = config.get('probability', 0.1)
        for i in range(len(chars) - 1):
            if rng.random() < prob and '\u0600' <= chars[i] <= '\u06FF' and '\u0600' <= chars[i+1] <= '\u06FF':
                chars[i], chars[i+1] = chars[i+1], chars[i]
        return ''.join(chars)

class SynonymReplacementAugmentation(AugmentationStrategy):
    def apply(self, text: str, config: Dict[str, Any], rng: np.random.Generator) -> str:
        words = text.split()
        synonym_map = config.get('synonym_map', {})
        prob = config.get('probability', 0.2)
        result = []
        for word in words:
            if word in synonym_map and rng.random() < prob:
                result.append(rng.choice(synonym_map[word]))
            else:
                result.append(word)
        return ' '.join(result)

class DynamicAugmenter:
    """Factory pattern for configurable augmentations"""

    STRATEGY_REGISTRY = {
        'vowel_deletion': VowelDeletionAugmentation(),
        'character_swap': CharacterSwapAugmentation(),
        'synonym_replacement': SynonymReplacementAugmentation()
    }

    def __init__(self, config: AugmentationConfig):
        self.config = config
        seed = config.strategies[0].get('seed', 42) if config.strategies else 42
        self.rng = np.random.default_rng(seed)

    def augment(self, text: str) -> str:
        result = text
        for strategy_cfg in self.config.strategies:
            if not strategy_cfg.get('enabled', True):
                continue

            name = strategy_cfg['name']
            if name in self.STRATEGY_REGISTRY:
                result = self.STRATEGY_REGISTRY[name].apply(
                    result,
                    {**strategy_cfg, 'synonym_map': self.config.synonym_map},
                    self.rng
                )
        return result


# ======================
# 4. DYNAMIC SUMMARIZATION PIPELINE
# ======================
class DynamicArabicSummarizer:
    """Fully configurable hybrid summarization"""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.preprocessor = DynamicArabicPreprocessor(config.normalization)
        self.augmenter = DynamicAugmenter(config.augmentation)
        self.device = self._detect_device(config.device)
        print(f"✓ Using device: {self.device}")
        self._model = None
        self._tokenizer = None
        self._embed_model = None
        self._embed_tokenizer = None

    def _detect_device(self, requested: str) -> str:
        if requested != "auto":
            return requested

        try:
            import torch
            if torch.cuda.is_available():
                return "cuda"
            elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
                return "mps"
        except ImportError:
            pass
        return "cpu"

    def _is_local_path(self, path: str) -> bool:
        """Safely check if model path is a local directory"""
        try:
            p = Path(path)
            return p.exists() and p.is_dir()
        except Exception:
            return False

    def _load_summarization_model(self):
        if self._model is not None:
            return

        try:
            from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
            import torch
        except ImportError as e:
            raise ImportError(
                "❌ Missing required packages. Install with:\n"
                "pip install transformers torch sentencepiece"
            ) from e

        model_id = self.config.abstractive.model_name
        is_local = self._is_local_path(model_id)

        print(f"✓ Loading abstractive model: {model_id} ({'local path' if is_local else 'HF Hub'})")

        try:
            self._tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                cache_dir=self.config.cache_dir,
                use_fast=True,
                trust_remote_code=False
            )
            self._model = AutoModelForSeq2SeqLM.from_pretrained(
                model_id,
                cache_dir=self.config.cache_dir,
                trust_remote_code=False
            ).to(self.device)
            self._model.eval()
            if self.device != "cpu" and torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            # Provide helpful error messages
            if "404" in str(e) or "Repository Not Found" in str(e):
                raise RuntimeError(
                    f"❌ Model '{model_id}' not found on Hugging Face Hub.\n"
                    f"   → Try one of these validated Arabic models:\n"
                    f"      • malmarjeh/t5-arabic-text-summarization (default)\n"
                    f"      • csebuetnlp/mT5_multilingual_XLSum\n"
                    f"      • UBC-NLP/t5-arabic-summarization"
                ) from e
            elif not is_local and "offline" in str(e).lower():
                raise RuntimeError(
                    f"❌ Cannot download model '{model_id}' (offline mode).\n"
                    f"   → Set cache_dir to a directory with pre-downloaded model, or go online."
                ) from e
            else:
                raise RuntimeError(
                    f"❌ Failed to load model '{model_id}':\n{str(e)[:300]}"
                ) from e

    def _load_embedding_model(self):
        if self._embed_model is not None:
            return

        try:
            from transformers import AutoTokenizer, AutoModel
            import torch
        except ImportError as e:
            raise ImportError(
                "❌ Missing required packages. Install with:\n"
                "pip install transformers torch scikit-learn"
            ) from e

        model_id = self.config.extractive.embedding_model
        is_local = self._is_local_path(model_id)

        print(f"✓ Loading embedding model: {model_id} ({'local path' if is_local else 'HF Hub'})")

        try:
            self._embed_tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                cache_dir=self.config.cache_dir,
                use_fast=True,
                trust_remote_code=False
            )
            self._embed_model = AutoModel.from_pretrained(
                model_id,
                cache_dir=self.config.cache_dir,
                trust_remote_code=False
            ).to(self.device)
            self._embed_model.eval()
            if self.device != "cpu" and torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            raise RuntimeError(
                f"❌ Failed to load embedding model '{model_id}':\n{str(e)[:300]}"
            ) from e

    def _get_embeddings(self, sentences: List[str]) -> np.ndarray:
        self._load_embedding_model()
        import torch

        embeddings = []
        for sent in sentences:
            inputs = self._embed_tokenizer(
                sent,
                return_tensors="pt",
                truncation=True,
                max_length=self.config.extractive.max_embedding_length,
                padding=True
            ).to(self.device)

            with torch.no_grad():
                outputs = self._embed_model(**inputs)
            # Use mean pooling instead of [CLS] for better semantic representation
            attention_mask = inputs['attention_mask']
            last_hidden = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            emb = (sum_embeddings / sum_mask).cpu().numpy()
            embeddings.append(emb)

        return np.vstack(embeddings)

    def _rank_sentences(self, sentences: List[str], embeddings: np.ndarray) -> List[str]:
        from sklearn.metrics.pairwise import cosine_similarity

        sim_matrix = cosine_similarity(embeddings)
        np.fill_diagonal(sim_matrix, 0)

        n = len(sentences)
        triangle_scores = np.zeros(n)

        # Triangle-aware centrality
        for i in range(n):
            for j in range(i + 1, n):
                if sim_matrix[i, j] > self.config.extractive.similarity_threshold:
                    common = np.where(
                        (sim_matrix[i] > self.config.extractive.similarity_threshold) &
                        (sim_matrix[j] > self.config.extractive.similarity_threshold)
                    )[0]
                    triangle_scores[i] += len(common)
                    triangle_scores[j] += len(common)

        degree_centrality = sim_matrix.sum(axis=1)
        combined = (
            self.config.extractive.degree_weight * degree_centrality +
            self.config.extractive.triangle_weight * triangle_scores
        )

        k = max(
            self.config.extractive.min_sentences,
            int(len(sentences) * self.config.extractive.ratio)
        )
        top_indices = np.argsort(combined)[::-1][:k]
        return [sentences[i] for i in sorted(top_indices)] if self.config.segmentation.preserve_order else [sentences[i] for i in top_indices]

    def _generate_summary(self, text: str) -> str:
        self._load_summarization_model()
        import torch

        # Dynamic prompt injection
        prompt = self.config.abstractive.prompt_template.format(
            text=text,
            **self.config.abstractive.prompt_params
        )

        inputs = self._tokenizer(
            prompt,
            return_tensors="pt",
            max_length=1024,
            truncation=True,
            padding="longest"
        ).to(self.device)

        with torch.no_grad():
            summary_ids = self._model.generate(
                inputs["input_ids"],
                max_length=self.config.abstractive.max_length,
                min_length=int(self.config.abstractive.max_length * self.config.abstractive.min_length_ratio),
                length_penalty=self.config.abstractive.length_penalty,
                num_beams=self.config.abstractive.num_beams,
                temperature=self.config.abstractive.temperature,
                no_repeat_ngram_size=self.config.abstractive.no_repeat_ngram_size,
                early_stopping=self.config.abstractive.early_stopping,
                do_sample=self.config.abstractive.temperature > 0.1
            )

        return self._tokenizer.decode(summary_ids[0], skip_special_tokens=True).strip()

    def _post_process(self, summary: str, original: str) -> str:
        cfg = self.config.postprocessing

        result = summary

        # Remove incomplete sentences
        if cfg.remove_incomplete_sentences and cfg.incomplete_sentence_pattern:
            result = re.sub(cfg.incomplete_sentence_pattern, '', result).strip()

        # Dialect to MSA conversion
        if cfg.dialect_to_msa:
            for dialect, msa in cfg.dialect_to_msa.items():
                result = result.replace(dialect, msa)

        # Punctuation fixes
        for fix in cfg.punctuation_fixes:
            result = re.sub(fix['pattern'], fix['replacement'], result)

        # Minimum length enforcement
        if len(result.split()) < cfg.min_summary_length and original:
            # Fallback: extract first N words from original
            words = original.split()[:cfg.min_summary_length + 5]
            result = ' '.join(words) + '...'

        return result.strip()

    def summarize(self, text: str, hybrid: Optional[bool] = None) -> Dict[str, Any]:
        # Resolve hybrid mode (config > override)
        use_hybrid = hybrid if hybrid is not None else self.config.hybrid_mode

        # Preprocessing
        clean_text = self.preprocessor.normalize(text)
        sentences = self.preprocessor.segment_sentences(clean_text, self.config.segmentation)

        # Edge case: empty text
        if not clean_text.strip():
            return {
                "summary": "",
                "method": "empty_input",
                "original_length": 0,
                "summary_length": 0,
                "compression_ratio": 0.0,
                "config_used": {"hybrid": use_hybrid}
            }

        # Method selection
        if len(sentences) < self.config.extractive.min_sentences or not use_hybrid:
            # Pure abstractive
            summary = self._generate_summary(clean_text)
            method = "pure_abstractive"
        else:
            # Hybrid pipeline
            embeddings = self._get_embeddings(sentences)
            extracted = self._rank_sentences(sentences, embeddings)
            extracted_text = " ".join(extracted)
            summary = self._generate_summary(extracted_text)
            method = "hybrid_extractive_abstractive"

        # Post-processing
        final_summary = self._post_process(summary, clean_text)

        return {
            "summary": final_summary,
            "method": method,
            "original_length": len(clean_text.split()),
            "summary_length": len(final_summary.split()),
            "compression_ratio": round(len(final_summary.split()) / max(1, len(clean_text.split())), 4),
            "config_used": {
                "hybrid": use_hybrid,
                "extractive_ratio": self.config.extractive.ratio,
                "max_length": self.config.abstractive.max_length
            }
        }

    def augment_and_summarize(self, text: str, num_variants: int = 3) -> Dict[str, Any]:
        """Generate multiple augmented variants and return best summary"""
        variants = [self.augmenter.augment(text) for _ in range(num_variants)]
        summaries = [self.summarize(v) for v in variants]

        # Simple scoring: prefer longer summaries (more informative) with proper punctuation
        scored = [
            (s, len(s['summary'].split()) * (1.2 if re.search(r'[.،!؟]', s['summary']) else 1.0))
            for s in summaries
        ]
        best = max(scored, key=lambda x: x[1])[0]

        return {
            **best,
            "augmentation_variants": num_variants,
            "all_summaries": [s['summary'] for s in summaries]
        }


# ======================
# 5. CONFIGURATION FACTORY & CLI
# ======================
def create_default_config() -> PipelineConfig:
    """Generate fully documented default configuration"""
    return PipelineConfig()

def save_default_config(path: str = "arabic_summarization_config.yaml"):
    """Save comprehensive default config with comments (YAML)"""
    config = create_default_config()

    # Build commented YAML manually for readability
    yaml_content = f"""# ARABIC SUMMARIZATION PIPELINE - FULLY CONFIGURABLE
# Every parameter can be modified without changing code

device: "auto"  # auto, cpu, cuda, mps
hybrid_mode: true
cache_dir: null
seed: 42

normalization:
  remove_diacritics: true
  diacritic_patterns:
    - "[\\u0617-\\u061A\\u064B-\\u0652\\u06D6-\\u06DC\\u06DF-\\u06E8\\u06EA-\\u06ED]"
  normalize_chars: true
  char_map:
    "أ": "ا"
    "إ": "ا"
    "آ": "ا"
    "ى": "ي"
    "ة": "ه"
    "گ": "ك"
    "ڤ": "ف"
    "پ": "ب"
    "چ": "ج"
  remove_tatweel: true
  tatweel_pattern: "ـ+"
  remove_latin: false
  extra_patterns: []

segmentation:
  delimiters: [".", "،", "!", "؟", "؛", ":"]
  min_sentence_length: 10
  preserve_order: true

extractive:
  ratio: 0.4
  min_sentences: 2
  similarity_threshold: 0.6
  triangle_weight: 0.3
  degree_weight: 0.7
  embedding_model: "UBC-NLP/MARBERTv2"
  max_embedding_length: 128

abstractive:
  # ✅ VALID PUBLIC MODEL (works out-of-the-box):
  model_name: "malmarjeh/t5-arabic-text-summarization"
  max_length: 150
  min_length_ratio: 0.3
  length_penalty: 1.2
  num_beams: 4
  temperature: 0.7
  no_repeat_ngram_size: 2
  early_stopping: true
  prompt_template: |
    النص: {{text}}

    التعليمات:
    1. حدد الموضوع الرئيسي
    2. استخرج الكيانات الأساسية
    3. اكتب ملخصًا موجزًا ({{min_sentences}}-{{max_sentences}} جمل) بالفصحى
    4. ركّز على الجوهر دون تفاصيل ثانوية

    الملخص:
  prompt_params:
    min_sentences: 2
    max_sentences: 3
  force_msa: true

augmentation:
  strategies:
    - name: "vowel_deletion"
      probability: 0.7
      enabled: true
    - name: "character_swap"
      probability: 0.1
      enabled: true
    - name: "synonym_replacement"
      probability: 0.2
      enabled: true
  synonym_map:
    "كبير": ["عظيم", "ضخم"]
    "صغير": ["ضئيل", "بسيط"]
    "جيد": ["ممتاز", "رائع"]
    "سيء": ["رديء", "ضعيف"]

postprocessing:
  remove_incomplete_sentences: true
  incomplete_sentence_pattern: "[\\u0600-\\u06FF]+[^\\u0600-\\u06FF.،!؟؛:]*$"
  dialect_to_msa:
    "عايز": "يريد"
    "بص": "انظر"
    "يالا": "هيّا"
  punctuation_fixes:
    - pattern: "\\s+([.،!؟؛:,])"
      replacement: "\\1"
    - pattern: "([.،!؟؛:,])\\s+"
      replacement: "\\1 "
  min_summary_length: 15
"""

    path_obj = Path(path)
    path_obj.parent.mkdir(parents=True, exist_ok=True)
    with open(path_obj, 'w', encoding='utf-8') as f:
        f.write(yaml_content)
    print(f"✓ Default config saved to: {path_obj.resolve()}")
    return str(path_obj)

def parse_cli_args():
    parser = argparse.ArgumentParser(description="Dynamic Arabic Summarization Pipeline")
    parser.add_argument("--config", type=str, default=None, help="Path to config file (YAML/JSON)")
    parser.add_argument("--text", type=str, default=None, help="Arabic text to summarize")
    parser.add_argument("--text-file", type=str, default=None, help="Path to file containing Arabic text")
    parser.add_argument("--output", type=str, default="summary.json", help="Output file path")
    parser.add_argument("--hybrid", type=lambda x: (str(x).lower() == 'true'), default=None, help="Force hybrid mode (true/false)")
    parser.add_argument("--augment", action="store_true", help="Apply augmentation before summarization")
    parser.add_argument("--save-default-config", action="store_true", help="Generate default config file")

    # Check if running in an interactive environment (like Colab or Jupyter)
    is_colab_or_jupyter = 'google.colab' in sys.modules or 'ipykernel_launcher.py' in sys.argv[0]

    if is_colab_or_jupyter:
        # Prevent argparse from parsing kernel arguments in notebooks/Colab
        args = parser.parse_args([])
    else:
        args = parser.parse_args()
    return args

def main():
    args = parse_cli_args()

    # Generate default config if requested
    if args.save_default_config:
        save_default_config()
        return

    # Load configuration
    if args.config:
        config = PipelineConfig.from_file(args.config)
        print(f"✓ Loaded config from: {args.config}")
    else:
        config = create_default_config()
        print("✓ Using default configuration (malmarjeh/t5-arabic-text-summarization)")

    # Initialize pipeline
    summarizer = DynamicArabicSummarizer(config)

    # Get input text
    if args.text_file:
        text_file = Path(args.text_file)
        if not text_file.exists():
            raise FileNotFoundError(f"Text file not found: {text_file}")
        with open(text_file, 'r', encoding='utf-8') as f:
            text = f.read().strip()
        print(f"✓ Loaded text from: {text_file}")
    elif args.text:
        text = args.text.strip()
        print("✓ Using inline text")
    else:
        # Demo mode with example text
        text = """
        أعلنت وزارة الصحة اليوم عن تسجيل 350 إصابة جديدة بفيروس كورونا المستجد خلال الـ24 ساعة الماضية،
        ليرتفع إجمالي عدد الإصابات في المملكة إلى 450 ألف حالة. وأشارت الوزارة في بيان صحفي إلى
        تعافي 280 مصابًا خلال اليوم نفسه، ليصل إجمالي المتعافين إلى 420 ألف حالة. كما سُجلت 4 وفيات
        جديدة، ليرتفع إجمالي الوفيات إلى 7500 حالة وفاة. وأهابت الوزارة بالمواطنين والمقيمين ضرورة
        الالتزام بالإجراءات الاحترازية والتباعد الاجتماعي وارتداء الكمامات في الأماكن المغلقة.
        """
        print("✓ Using demo text (provide --text or --text-file for custom input)")

    # Summarize
    print("\n🔄 Generating summary...")
    try:
        if args.augment:
            result = summarizer.augment_and_summarize(text)
            print("✓ Used augmentation ensemble")
        else:
            result = summarizer.summarize(text, hybrid=args.hybrid)
            print(f"✓ Used method: {result['method']}")
    except Exception as e:
        print(f"\n❌ Summarization failed: {e}")
        print("\n💡 TIPS:")
        print("   • For first run: Ensure internet connection to download models")
        print("   • Install requirements: pip install transformers torch sentencepiece scikit-learn")
        print("   • Use --save-default-config to generate a config file for customization")
        sys.exit(1)

    # Output
    print("\n" + "="*70)
    print("SUMMARY RESULT")
    print("="*70)
    print(result['summary'])
    print("="*70)
    print(f"Stats: {result['summary_length']} words | Compression: {result['compression_ratio']*100:.1f}%")

    # Save to file
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"\n✓ Full result saved to: {output_path.resolve()}")


# ======================
# 6. QUICK START EXAMPLES
# ======================
def example_usage():
    """Demonstrate dynamic configuration in code"""

    # Example 1: Minimal config override
    config1 = PipelineConfig(
        abstractive=AbstractiveGenerationConfig(
            max_length=100,
            temperature=0.9  # more creative
        ),
        hybrid_mode=False  # pure abstractive
    )

    # Example 2: Dialect-heavy social media config
    config2 = PipelineConfig(
        normalization=ArabicNormalizationConfig(
            remove_diacritics=True,
            normalize_chars=True,
            remove_latin=True  # remove English words/emojis
        ),
        postprocessing=PostProcessingConfig(
            dialect_to_msa={
                'عايز': 'يريد', 'بتاع': 'خاصة', 'كده': 'هكذا',
                'مفيش': 'لا يوجد', 'بقى': 'أصبح'
            },
            min_summary_length=10
        ),
        abstractive=AbstractiveGenerationConfig(
            prompt_template="لخص المنشور التالي بلغة عامية مفهومة:\nالنص: {text}\nالملخص:",
            force_msa=False
        )
    )

    # Example 3: Legal/document config (precision-focused)
    config3 = PipelineConfig(
        extractive=ExtractiveRankingConfig(
            ratio=0.6,  # keep more content
            similarity_threshold=0.75,  # stricter similarity
            triangle_weight=0.5  # emphasize discourse structure
        ),
        abstractive=AbstractiveGenerationConfig(
            num_beams=8,  # higher quality
            temperature=0.3,  # less random
            length_penalty=2.0  # prefer longer outputs
        )
    )

    print("✓ Config examples created dynamically")
    return config1, config2, config3


if __name__ == "__main__":
    main()

    # For interactive use:
    # configs = example_usage()
    # summarizer = DynamicArabicSummarizer(configs[0])
    # result = summarizer.summarize("نص عربي للتلخيص...")

✓ Using default configuration (malmarjeh/t5-arabic-text-summarization)
✓ Using device: cpu
✓ Using demo text (provide --text or --text-file for custom input)

🔄 Generating summary...
✓ Loading embedding model: UBC-NLP/MARBERTv2 (HF Hub)
✓ Loading abstractive model: /content/drive/MyDrive/summerize/t5-arabic-hybrid-summarizer (local path)


The tokenizer you are loading from '/content/drive/MyDrive/summerize/t5-arabic-hybrid-summarizer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✓ Used method: hybrid_extractive_abstractive

SUMMARY RESULT
تعافي 280 حالة وتعافي 280 في اليوم الثاني من الإصابة ب " كورونا " خلال اليوم الأول من اليوم الحالي في الأردن والتعافي 2. 5 ألف حالة في يوم واحد و 5 أيام من العلاج والعلاج في أبوظبي
Stats: 38 words | Compression: 52.8%

✓ Full result saved to: /content/summary.json
